# Gemma3 Zero-Shot: Headlines → 0 (Up) / 1 (Down)

Calls the Pitt API with Gemma3-4B.  
Same data pipeline as `combination_multisent.ipynb` (MOVE_THRESHOLD=0.2%, intraday target).  
Gemma3 predicts **0** (close ≥ open) or **1** (close < open) from raw headlines — no fine-tuning.

In [1]:
import pandas as pd
import numpy as np
import json, os, shutil
from tqdm import tqdm
from openai import OpenAI
from time import sleep
from sklearn.metrics import (
    accuracy_score, classification_report,
    precision_score, recall_score, f1_score, confusion_matrix
)
from google.colab import files

WORK_DIR   = '/content'
DATA_FILE  = f'{WORK_DIR}/sp500_headlines_with_prices sep ~.csv'
CACHE_FILE = f'{WORK_DIR}/gemma_cache.json'

# Load API key from config.py (never committed to GitHub)
if not os.path.exists(f'{WORK_DIR}/config.py'):
    print('Please upload config.py')
    uploaded = files.upload()
    for fn in uploaded:
        shutil.move(fn, f'{WORK_DIR}/config.py')

import importlib.util
spec = importlib.util.spec_from_file_location('config', f'{WORK_DIR}/config.py')
cfg  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cfg)

LLM_API_KEY  = cfg.LLM_API_KEY
API_BASE_URL = cfg.API_BASE_URL
GEMMA_MODEL  = cfg.GEMMA_MODEL
print(f'Model: {GEMMA_MODEL}  |  Base URL: {API_BASE_URL}')

Please upload config.py


Saving config.py to config.py
Model: gemma3  |  Base URL: https://ol.sci.pitt.edu


In [2]:
# Upload & load data
if not os.path.exists(DATA_FILE):
    print('Please upload: sp500_headlines_with_prices sep ~.csv')
    uploaded = files.upload()
    for fn in uploaded:
        shutil.move(fn, DATA_FILE)

df = pd.read_csv(DATA_FILE)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# 0 = up (close >= open), 1 = down — matches prompt wording
df['target'] = (df['close_price'] < df['open_price']).astype(int)

MOVE_THRESHOLD = 0.002
df['daily_move'] = (df['close_price'] - df['open_price']).abs() / df['open_price']
before = len(df)
df = df[df['daily_move'] > MOVE_THRESHOLD].reset_index(drop=True)
print(f'Rows: {before} -> {len(df)} after removing near-flat days')
print(f'Target dist (0=up, 1=down): {df["target"].value_counts().sort_index().to_dict()}')

split_index = int(len(df) * 0.8)
train_df = df.iloc[:split_index].copy()
test_df  = df.iloc[split_index:].copy()
print(f'Train: {len(train_df)}  |  Test: {len(test_df)}')

Please upload: sp500_headlines_with_prices sep ~.csv


Saving sp500_headlines_with_prices sep ~.csv to sp500_headlines_with_prices sep ~.csv
Rows: 3507 -> 2572 after removing near-flat days
Target dist (0=up, 1=down): {0: 1425, 1: 1147}
Train: 2057  |  Test: 515


In [3]:
def build_prompt(headlines):
    return (
        'You are a financial market analyst.\n\n'
        'Below are S&P 500 news headlines from one trading day.\n'
        'Predict whether the S&P 500 will close HIGHER or LOWER than its opening price that same day.\n\n'
        'Reply with a single digit only — no spaces, no punctuation, no other text:\n'
        '0  means the market closes UP (close >= open)\n'
        '1  means the market closes DOWN (close < open)\n\n'
        f'Headlines:\n{headlines}\n\nAnswer:'
    )


if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE) as f:
        cache = json.load(f)
    print(f'Loaded cache: {len(cache)} entries')
else:
    cache = {}
    print('No cache found, starting fresh.')


def run_gemma(headlines):
    prompt    = build_prompt(headlines)
    cache_key = prompt
    if cache_key in cache:
        return cache[cache_key]

    client = OpenAI(api_key=LLM_API_KEY, base_url=API_BASE_URL, timeout=30.0)
    response_text = None
    for attempt, wait in enumerate([0, 6, 15]):
        try:
            resp = client.chat.completions.create(
                model=GEMMA_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0,
                max_tokens=5,
            )
            response_text = resp.choices[0].message.content.strip()
            break
        except Exception as e:
            print(f'  Retry {attempt+1}/3: {e}')
            if wait:
                sleep(wait)

    cache[cache_key] = response_text
    with open(CACHE_FILE, 'w') as f:
        json.dump(cache, f)
    return response_text


def parse_output(text):
    if text is None:
        return None
    t = text.strip()
    if '0' in t:
        return 0
    elif '1' in t:
        return 1
    return None


print('Prompt and API wrapper ready.')

No cache found, starting fresh.
Prompt and API wrapper ready.


In [4]:
# Inference loop on test set
preds       = []
labels      = []
raw_outputs = []
skipped     = 0

for i in tqdm(range(len(test_df))):
    row       = test_df.iloc[i]
    headlines = str(row['combined_headline'])

    raw  = run_gemma(headlines)
    pred = parse_output(raw)
    raw_outputs.append(raw)

    if pred is not None:
        preds.append(pred)
        labels.append(int(row['target']))
    else:
        skipped += 1
        print(f"  [unparseable] idx={i}  raw='{raw}'")

print(f'\nDone. Parsed: {len(preds)}/{len(test_df)}  |  Skipped (unparseable): {skipped}')

  8%|▊         | 40/515 [00:57<07:13,  1.10it/s]

  Retry 1/3: Request timed out.
  Retry 2/3: Request timed out.


 33%|███▎      | 171/515 [09:17<05:56,  1.04s/it]

  Retry 1/3: Request timed out.


 50%|████▉     | 255/515 [15:02<1:24:28, 19.50s/it]

  Retry 1/3: Request timed out.


100%|██████████| 515/515 [27:06<00:00,  3.16s/it]


Done. Parsed: 515/515  |  Skipped (unparseable): 0


In [5]:
# Print Gemma raw answers
print(f'{"idx":>4}  {"raw answer":>12}  {"parsed":>8}  {"true label":>10}  {"correct?":>9}')
print('-' * 52)
for i, (raw, pred, lbl) in enumerate(zip(raw_outputs, preds, labels)):
    correct = 'YES' if pred == lbl else 'NO'
    print(f'{i:>4}  {str(raw):>12}  {str(pred):>8}  {lbl:>10}  {correct:>9}')

 idx    raw answer    parsed  true label   correct?
----------------------------------------------------
   0             1         1           1        YES
   1             0         0           0        YES
   2             0         0           0        YES
   3             0         0           0        YES
   4             0         0           0        YES
   5             0         0           1         NO
   6             0         0           0        YES
   7             0         0           1         NO
   8             1         1           0         NO
   9             0         0           1         NO
  10             0         0           0        YES
  11             0         0           0        YES
  12             1         1           1        YES
  13             1         1           1        YES
  14             1         1           0         NO
  15             1         1           0         NO
  16             0         0           0        YES
  17       

In [6]:
# Evaluate
tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

print('=== Gemma3-4B Zero-Shot ===')
print(classification_report(labels, preds, target_names=['0 (Up)', '1 (Down)']))
print(f'Accuracy  : {accuracy_score(labels, preds):.4f}')
print(f'Precision : {precision_score(labels, preds):.4f}')
print(f'Recall    : {recall_score(labels, preds):.4f}')
print(f'F1-score  : {f1_score(labels, preds):.4f}')
print(f'TP: {tp}  TN: {tn}  FP: {fp}  FN: {fn}')

acc_gemma = accuracy_score(labels, preds)
print('\n======================================================')
print(f'  Random baseline (always up)               : 53.5%')
print(f'  StockPredict   FinBERT -> LinearSVC        : 64.1%')
print(f'  Combination    LinearSVC  (7 features)    : 64.2%')
print(f'  MultiSent      Transformer (11 features)  : 65.0%')
print(f'  Gemma3-4B      Zero-Shot                  : {acc_gemma:.1%}')
print('======================================================')

=== Gemma3-4B Zero-Shot ===
              precision    recall  f1-score   support

      0 (Up)       0.84      0.63      0.72       276
    1 (Down)       0.67      0.86      0.75       239

    accuracy                           0.74       515
   macro avg       0.75      0.75      0.74       515
weighted avg       0.76      0.74      0.74       515

Accuracy  : 0.7379
Precision : 0.6699
Recall    : 0.8577
F1-score  : 0.7523
TP: 205  TN: 175  FP: 101  FN: 34

  Random baseline (always up)               : 53.5%
  StockPredict   FinBERT -> LinearSVC        : 64.1%
  Combination    LinearSVC  (7 features)    : 64.2%
  MultiSent      Transformer (11 features)  : 65.0%
  Gemma3-4B      Zero-Shot                  : 73.8%


Zero-shot LLM (Gemma3-4B) outperforms all supervised models including FinBERT+Transformer by ~9%, suggesting that general language understanding from large-scale pretraining captures financial signals more effectively than task-specific fine-tuning on limited data.